# 01. Контекстный пакет

В этом мини-примере агент не выглядит как «живое существо». Это обычный харнес вокруг LLM: он собирает нужный контекст, передает его в модель и потом разбирает ответ.

Мини-лор: **Northstar Compute** — вымышленная AI-инфраструктурная компания из кэпстоун-проекта. Она продаёт управляемую GPU-capacity. **Aurora-17** — учебный инцидент в одном из её GPU-кластеров. Глубоко знать этот мир пока не нужно: здесь он нужен только как живой домен для примера.

Наша задача: руками собрать маленький пакет контекста для будущего LLM Wiki Agent.

In [ ]:
from pprint import pprint


context_packet = {
    "task": "Обновить wiki-страницу о клиентском инциденте Aurora-17",
    "instructions": [
        "Пиши кратко и проверяемо.",
        "Не добавляй факты без источника.",
        "Если источник противоречит wiki, пометь конфликт явно.",
    ],
    "sections": [
        {
            "name": "run_state",
            "trust": "internal",
            "content": "run_id=demo-001; step=curate; milestone=M2",
        },
        {
            "name": "current_wiki_page",
            "trust": "workspace",
            "content": "Aurora-17 был сетевым инцидентом в кластере EU-West.",
        },
        {
            "name": "new_source_excerpt",
            "trust": "untrusted_source",
            "content": "Новый postmortem говорит, что корневой причиной был лимит очереди jobs.",
        },
    ],
}

pprint(context_packet, sort_dicts=False)

Главная мысль: **память агента не возникает сама**. Харнес явно решает, что положить в следующий вызов модели: инструкции, состояние запуска, куски wiki, новые источники и ограничения.

In [ ]:
def render_context(packet: dict) -> str:
    lines = [
        "# Задача",
        packet["task"],
        "",
        "# Правила",
    ]
    lines.extend(f"- {item}" for item in packet["instructions"])
    lines.append("")

    for section in packet["sections"]:
        lines.append(f"# {section['name']} [{section['trust']}]")
        lines.append(section["content"])
        lines.append("")

    return "\n".join(lines).strip()


print(render_context(context_packet))